# 15.10 Bandit-based recommendation

Bandits recommend while learning, paying some short-term regret to discover better long-term choices.

This notebook implements real NumPy epsilon-greedy, UCB, and Thompson sampling logic. The D5 rung includes eligibility guardrails, drift, and long-tail items so exploration remains useful but safe.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(1507)


def softmax(scores, temperature=1.0):
    values = np.asarray(scores, dtype=float) / float(temperature)
    values = values - np.max(values)
    weights = np.exp(values)
    return weights / np.sum(weights)


def normalize_rows(matrix):
    values = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return values / norms


def top_k_indices(scores, k):
    values = np.asarray(scores, dtype=float)
    order = np.argsort(-values)
    return order[:k]


def recall_at_k(scores, positives, k=3):
    values = np.asarray(scores, dtype=float)
    hits = []
    for row, pos in zip(values, positives):
        top = set(top_k_indices(row, k))
        wanted = set(np.atleast_1d(pos).astype(int))
        hits.append(len(top & wanted) / max(1, len(wanted)))
    return float(np.mean(hits))


def dcg_at_k(relevance, k=3):
    rel = np.asarray(relevance, dtype=float)[:k]
    gains = np.power(2.0, rel) - 1.0
    discounts = np.log2(np.arange(2, len(rel) + 2))
    return float(np.sum(gains / discounts))


def ndcg_at_k(scores, relevance, k=3):
    values = np.asarray(scores, dtype=float)
    rel = np.asarray(relevance, dtype=float)
    order = np.argsort(-values)[:k]
    ideal = np.argsort(-rel)[:k]
    ideal_dcg = dcg_at_k(rel[ideal], k)
    if ideal_dcg == 0.0:
        return 0.0
    return dcg_at_k(rel[order], k) / ideal_dcg


def mrr_from_scores(scores, target):
    order = np.argsort(-np.asarray(scores, dtype=float))
    rank = int(np.where(order == int(target))[0][0]) + 1
    return 1.0 / rank


def make_f14_ladder(seed=1514):
    rng = np.random.default_rng(seed)
    rungs = []

    item_vectors = np.array([
        [1.0, 0.1, 0.0],
        [0.8, 0.4, 0.0],
        [0.1, 1.0, 0.2],
        [-0.2, 0.3, 0.9],
    ])
    user_vectors = np.array([
        [1.0, 0.2, 0.0],
        [0.2, 1.0, 0.2],
        [-0.1, 0.4, 1.0],
    ])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D1 tiny slate",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.array([0]), np.array([2]), np.array([3])],
        "sessions": [[0, 1, 2], [1, 2, 0], [2, 3, 1]],
        "targets": [2, 0, 1],
        "clicks": np.array([5, 8, 1]),
        "impressions": np.array([100, 100, 20]),
        "eligible": np.array([True, True, True, True]),
        "exposure": np.array([1.0, 0.8, 0.5, 0.3]),
    })

    angles = np.linspace(0.0, 2.0 * np.pi, 8, endpoint=False)
    item_vectors = np.column_stack([np.cos(angles), np.sin(angles), 0.35 * np.cos(2.0 * angles)])
    user_angles = np.array([0.1, 1.6, 3.0, 4.7, 5.6])
    user_vectors = np.column_stack([np.cos(user_angles), np.sin(user_angles), 0.25 * np.sin(user_angles)])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D2 synthetic preferences",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 1, 2, 3], [2, 3, 4], [4, 5, 6], [6, 7, 0], [7, 0, 1]],
        "targets": [3, 4, 6, 0, 1],
        "clicks": np.array([12, 20, 18, 6, 4, 3, 9, 7]),
        "impressions": np.array([200, 260, 220, 150, 80, 60, 120, 140]),
        "eligible": np.ones(8, dtype=bool),
        "exposure": np.linspace(1.0, 0.35, 8),
    })

    item_vectors = rng.normal(size=(12, 4))
    item_vectors = normalize_rows(item_vectors)
    user_vectors = rng.normal(size=(7, 4))
    user_vectors = normalize_rows(user_vectors)
    relevance = user_vectors @ item_vectors.T
    relevance = relevance + rng.normal(scale=0.04, size=relevance.shape)
    exposure = rng.uniform(0.15, 1.0, size=12)
    observed = rng.uniform(size=relevance.shape) < exposure[None, :]
    sparse_relevance = np.where(observed, relevance, -0.2)
    rungs.append({
        "name": "D3 sparse noisy logs",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": sparse_relevance,
        "true_relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 2, 5, 7], [1, 2, 4], [3, 8, 9], [10, 2, 0], [6, 6, 11], [4, 5, 6], [9, 1, 3]],
        "targets": [7, 4, 9, 0, 11, 6, 3],
        "clicks": np.array([30, 16, 9, 4, 3, 2, 8, 7, 5, 4, 2, 1]),
        "impressions": np.array([600, 280, 190, 90, 45, 30, 120, 110, 70, 65, 22, 12]),
        "eligible": rng.uniform(size=12) > 0.10,
        "exposure": exposure,
    })

    genres = np.array([
        [1, 0, 0, 0, 1],
        [1, 1, 0, 0, 0],
        [0, 1, 1, 0, 0],
        [0, 0, 1, 1, 0],
        [0, 0, 0, 1, 1],
        [1, 0, 0, 1, 0],
        [0, 1, 0, 0, 1],
        [0, 0, 1, 0, 1],
        [1, 0, 1, 0, 0],
        [0, 1, 0, 1, 0],
    ], dtype=float)
    item_vectors = normalize_rows(genres + 0.05 * rng.normal(size=genres.shape))
    user_vectors = normalize_rows(np.array([
        [2, 0, 0, 0, 1],
        [1, 2, 0, 0, 0],
        [0, 1, 2, 0, 0],
        [0, 0, 1, 2, 0],
        [0, 0, 0, 1, 2],
        [1, 0, 1, 0, 1],
    ], dtype=float))
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D4 MovieLens-like inline",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 5, 8], [1, 2, 6], [2, 3, 8], [3, 4, 9], [4, 7, 0], [7, 8, 6]],
        "targets": [8, 6, 3, 9, 4, 6],
        "clicks": np.array([50, 34, 25, 18, 17, 14, 9, 7, 6, 5]),
        "impressions": np.array([1000, 700, 520, 430, 320, 250, 150, 110, 90, 80]),
        "eligible": np.ones(10, dtype=bool),
        "exposure": np.array([1.0, 0.9, 0.75, 0.65, 0.55, 0.45, 0.35, 0.30, 0.25, 0.20]),
    })

    head = normalize_rows(rng.normal(loc=0.6, scale=0.25, size=(5, 5)))
    torso = normalize_rows(rng.normal(loc=0.0, scale=0.7, size=(8, 5)))
    cold = normalize_rows(np.array([
        [1, 1, 0, 0, 1],
        [0, 1, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [0, 0, 1, 1, 1],
    ], dtype=float))
    item_vectors = np.vstack([head, torso, cold])
    user_vectors = normalize_rows(rng.normal(size=(8, 5)))
    user_vectors[0] = normalize_rows(np.array([[1, 1, 0, 0, 1]], dtype=float))[0]
    relevance = user_vectors @ item_vectors.T
    popularity = np.array([2000, 1500, 1200, 900, 700, 250, 180, 130, 100, 75, 60, 45, 35, 10, 8, 5, 3])
    clicks = np.maximum(1, np.round(popularity * np.clip(0.03 + 0.04 * np.maximum(item_vectors[:, 0], 0.0), 0.01, 0.20))).astype(int)
    exposure = np.clip(popularity / popularity.max(), 0.02, 1.0)
    rungs.append({
        "name": "D5 long-tail cold-start",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 1, 14], [1, 6, 15], [2, 8, 16], [3, 9, 13], [4, 10, 14], [5, 11, 15], [6, 12, 16], [7, 13, 14]],
        "targets": [14, 15, 16, 13, 14, 15, 16, 14],
        "clicks": clicks,
        "impressions": popularity.astype(int),
        "eligible": np.array([True] * 13 + [True, True, False, True]),
        "exposure": exposure,
        "cold_items": np.array([13, 14, 15, 16]),
    })

    return rungs


## The concept, built once: choose arms under uncertainty

Epsilon-greedy gives exploration mass to every eligible arm. UCB chooses

$$a_t=\arg\max_a \left(\hat\mu_a+\sqrt{\frac{2\ln t}{n_a}}\right),$$

and Thompson sampling samples from each arm's Beta posterior.

In [ ]:

def bandit_recommend(means, pulls, clicks=None, nonclicks=None, epsilon=0.2, t=None, method="ucb", eligible=None, rng=None):
    values = np.asarray(means, dtype=float)
    counts = np.asarray(pulls, dtype=float)
    if eligible is None:
        eligible = np.ones_like(values, dtype=bool)
    eligible = np.asarray(eligible, dtype=bool)
    if rng is None:
        rng = np.random.default_rng(1510)
    if method == "epsilon_greedy":
        probabilities = np.zeros_like(values, dtype=float)
        choices = np.flatnonzero(eligible)
        best = choices[np.argmax(values[choices])]
        probabilities[choices] = epsilon / len(choices)
        probabilities[best] += 1.0 - epsilon
        return {"scores": probabilities, "choice": int(best)}
    if method == "ucb":
        if t is None:
            t = float(np.sum(counts))
        bonus = np.sqrt(2.0 * math.log(float(t)) / np.maximum(counts, 1.0))
        scores = values + bonus
        scores = np.where(eligible, scores, -np.inf)
        return {"scores": scores, "choice": int(np.argmax(scores))}
    if method == "thompson":
        alpha = np.asarray(clicks, dtype=float) + 1.0
        beta = np.asarray(nonclicks, dtype=float) + 1.0
        samples = rng.beta(alpha, beta)
        samples = np.where(eligible, samples, -np.inf)
        means = alpha / (alpha + beta)
        return {"scores": samples, "posterior_means": means, "choice": int(np.argmax(samples))}
    raise ValueError("unknown method")


def cumulative_regret(best_reward, chosen_rewards):
    regrets = []
    total = 0.0
    for reward in chosen_rewards:
        total += float(best_reward) - float(reward)
        regrets.append(total)
    return np.array(regrets)


## Verify the lesson numbers once

With $\varepsilon=0.2$ over three arms, the best arm gets $0.867$ and the other arms get $0.067$. UCB scores are $0.361$, $0.775$, and $1.430$. Thompson posterior means are $0.059$, $0.088$, and $0.091$. Rewards $(0.05,0.08,0.04,0.10,0.08)$ against best reward $0.10$ end with cumulative regret $0.15$.

In [ ]:

means = np.array([0.05, 0.08, 0.04])
pulls = np.array([100, 20, 5])
eps = bandit_recommend(means, pulls, epsilon=0.2, method="epsilon_greedy")
assert [round(float(x), 3) for x in eps["scores"]] == [0.067, 0.867, 0.067]

ucb = bandit_recommend(means, pulls, t=125, method="ucb")
assert [round(float(x), 3) for x in ucb["scores"]] == [0.361, 0.775, 1.430]
assert ucb["choice"] == 2

thompson = bandit_recommend(means, pulls, clicks=np.array([5, 8, 1]), nonclicks=np.array([95, 92, 19]), method="thompson")
assert [round(float(x), 3) for x in thompson["posterior_means"]] == [0.059, 0.088, 0.091]

regret = cumulative_regret(0.10, [0.05, 0.08, 0.04, 0.10, 0.08])
assert [round(float(x), 2) for x in regret] == [0.05, 0.07, 0.13, 0.13, 0.15]
print("epsilon", np.round(eps["scores"], 3))
print("ucb", np.round(ucb["scores"], 3), "regret", np.round(regret, 2))


## The dataset ladder: D1 to D5

The same F14 recommender ladder is built inline: tiny slate, synthetic preferences, sparse noisy logs, a MovieLens-like genre matrix, and a long-tail cold-start catalog. The single metric here is **recall@3 with regret**.

In [ ]:

rungs = make_f14_ladder()
for rung in rungs:
    user_shape = rung["user_vectors"].shape
    item_shape = rung["item_vectors"].shape
    rel_shape = rung["relevance"].shape
    sparsity = float(np.mean(rung.get("exposure", np.ones(item_shape[0])) < 0.5))
    print(rung["name"], "users", user_shape, "items", item_shape, "relevance", rel_shape, "low exposure fraction", round(sparsity, 3))
    print("sample relevance row", np.round(rung["relevance"][0, : min(5, item_shape[0])], 3))


## Run the same method across D1-D5

Each rung uses UCB scores over arms/items. Recall@3 checks whether relevant arms are in the recommendation slate, while regret records the cost of not always choosing the highest expected reward.

In [ ]:

results = []
for rung in rungs:
    impressions = rung["impressions"].astype(float)
    clicks = rung["clicks"].astype(float)
    means = clicks / np.maximum(impressions, 1.0)
    output = bandit_recommend(means, impressions, t=float(np.sum(impressions)), method="ucb", eligible=rung["eligible"])
    scores = output["scores"][None, :]
    positives = [rung["positives"][0]]
    metric = recall_at_k(scores, positives, k=3)
    top_rewards = means[np.argsort(-output["scores"])[:5]]
    best_reward = float(np.max(means[rung["eligible"]]))
    regret = cumulative_regret(best_reward, top_rewards)
    results.append({"name": rung["name"], "metric": metric, "scores": output["scores"], "regret": regret})

for row in results:
    print(f"{row['name']:<28} recall@3={row['metric']:.3f} final_regret={row['regret'][-1]:.3f}")


## Results visualization

Panels show UCB arm scores per rung; the curve shows recall@3 with the final regret annotated in the title.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()
for ax, rung, row in zip(flat_axes[:5], rungs, results):
    finite_scores = np.where(np.isfinite(row["scores"]), row["scores"], np.nan)
    order = np.argsort(-np.nan_to_num(finite_scores, nan=-999.0))[:5]
    ax.bar(np.arange(len(order)), finite_scores[order], color="mediumseagreen")
    ax.set_title(rung["name"])
    ax.set_xlabel("arm rank")
    ax.set_ylabel("UCB score")
flat_axes[5].plot(range(1, 6), [row["metric"] for row in results], marker="o")
flat_axes[5].set_xticks(range(1, 6))
flat_axes[5].set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
flat_axes[5].set_ylim(0.0, 1.05)
flat_axes[5].set_title(f"recall@3 curve, D5 regret={results[-1]['regret'][-1]:.3f}")
fig.tight_layout()


## Pitfall on D5: unsafe exploration and drift

Exploring every arm ignores product guardrails, and stationary counts can miss drift. The fix filters ineligible arms and uses a recency-window estimate before UCB ranking.

In [ ]:

d5 = rungs[-1]
means = d5["clicks"] / np.maximum(d5["impressions"], 1)
unsafe = bandit_recommend(means, d5["impressions"], t=float(np.sum(d5["impressions"])), method="ucb", eligible=np.ones_like(d5["eligible"], dtype=bool))
recent_clicks = np.maximum(1.0, d5["clicks"] * np.linspace(0.7, 1.4, len(d5["clicks"])))
recent_impressions = np.maximum(5.0, d5["impressions"] * np.linspace(0.4, 0.1, len(d5["impressions"])))
recent_means = recent_clicks / recent_impressions
safe = bandit_recommend(recent_means, recent_impressions, t=float(np.sum(recent_impressions)), method="ucb", eligible=d5["eligible"])
print("unsafe top arms", np.argsort(-unsafe["scores"])[:5])
print("safe recency-window top arms", np.argsort(-safe["scores"])[:5])
print("ineligible selected unsafely", bool(np.argsort(-unsafe["scores"])[0] in np.flatnonzero(~d5["eligible"])))


## Evaluate it + practice

- Report **recall@3 and cumulative regret** against a no-skill popularity or random baseline on every rung.
- Sanity check that the D1 worked numbers match the lesson asserts before trusting larger rungs.
- Ablation: remove eligibility filters or use all-time counts under drift; the metric should drop on D3-D5.
- Failure signals: head-item collapse, cold items never appearing, or a metric curve that improves only because labels are biased.
- Keep splits chronological or exposure-aware when the lesson's pitfall involves time or logging.

Practice: change the cutoff from 3 to 5 and compare the metric curve.

Practice: compare epsilon-greedy and Thompson sampling on D5 with the same eligibility mask.